# agentorch 实验手册（完整功能版）

这个 notebook 基于当前最新的 `agentorch` 代码重写，目标是把**当前已经支持的全部核心能力**都整理成可以直接实验的示例。

你可以把它当成：

1. 快速上手手册
2. 功能验证脚本
3. 架构设计示例集

当前覆盖能力包括：

- OpenAI-compatible 模型接入
- 单 agent 基础运行
- 结构化工具注册与调用
- 代码解释器与 sandbox 执行
- memory 与 thread 清理
- workflow 基础编排
- RAG-ready 检索接口与最小本地实现
- AgentRegistry 注册中心
- supervisor 多智能体动态委派
- workflow 中的 agent 节点


## 0. 先看最重要的使用规则

在 Jupyter / Notebook 里，请优先使用新版异步创建接口：

- 不要调用 `agent.run_sync(...)`
- 不要在 notebook 里调用 `Runtime.create(...)` / `Agent.create(...)` / `IndexedKnowledgeBase.create(...)`
- notebook / Jupyter 环境请使用：`await Runtime.acreate(...)`、`await Agent.acreate(...)`、`await IndexedKnowledgeBase.acreate(...)`
- 如果修改了本地包代码，请重启 kernel 后重新运行
- 涉及 tool calling、workflow、多 agent、memory、RAG 的实验时，建议每个示例使用独立 `thread_id`

当前 notebook 已统一改成**最新推荐 API 风格**：

- `RuntimeConfig.agent(...)` / `RuntimeConfig.workflow(...)`
- `orchestration_profile='default_safe' / 'deep_research' / 'research_heavy'`
- `context_strategy / long_horizon_strategy / cooperation_strategy / memory_governance_strategy`
- `RagStrategyConfig.for_classic(...) / for_deliberative(...) / for_hybrid(...)`
- `ReasoningStrategyConfig.react(...) / plan_execute(...) / reflexion(...)`
- `ToolRegistry.from_tools(...) / ToolRegistry.with_bundles(...)`
- `WorkflowBuilder()` + `Node.*(...)` shortcuts
- `DeepResearchAgent.acreate(...)` 作为研究型 preset 入口


## 1. 环境准备

推荐 Python 版本：`3.11+`

如果你和当前项目保持一致，优先使用现在这套环境：`data_analysis_py311`。

推荐安装命令：

```powershell
python -m pip install -e .
python -m pip install jupyterlab notebook
```

如果你要和命令行回归测试保持一致，也可以显式使用：

```powershell
C:\Users\24260\.conda\envs\data_analysis_py311\python.exe -m pip install -e .
```


In [4]:
import sys
from pathlib import Path

print(sys.version)
print(Path.cwd())


3.11.13 | packaged by conda-forge | (main, Jun  4 2025, 14:39:58) [MSC v.1943 64 bit (AMD64)]
c:\Users\24260\Desktop\研究生生涯\智能体开发范式


## 2. API Key / Base URL 导入方式

`agentorch` 会自动读取项目根目录下的 `.env`，并兼容两套变量名：

- `OPENAI_API_KEY` / `OPENAI_BASE_URL`
- `API_KEY` / `BASE_URL`

推荐写法：

```env
OPENAI_API_KEY=sk-xxxx
OPENAI_BASE_URL=https://api.openai.com/v1
```

如果你用 OpenAI-compatible 代理，也支持：

```env
API_KEY=sk-xxxx
BASE_URL=https://your-proxy.example.com/v1/chat/completions
```

框架会自动把 `.../chat/completions` 规整为 SDK 所需的 `/v1` 根地址。


In [5]:
from agentorch.config import ModelConfig

cfg = ModelConfig.from_any('gpt-4.1-mini', temperature=0)
print('api_key_loaded:', bool(cfg.api_key))
print('base_url:', cfg.base_url)
print('default_model:', cfg.model)
print('timeout:', cfg.timeout)



api_key_loaded: True
base_url: https://www.dmxapi.cn/v1
default_model: gpt-4.1-mini
timeout: 60.0


## 3. 当前公开 API 与策略入口

这一节建议重点关注三类统一入口：

- 顶层能力导出：`agentorch.__all__`
- 统一配置入口：`RuntimeConfig.agent(...)` / `RuntimeConfig.workflow(...)`
- 可选策略入口：`list_orchestration_profiles()` / `list_context_strategies()` / `list_cooperation_strategies()`


In [6]:
import agentorch
from agentorch import (
    list_context_strategies,
    list_cooperation_strategies,
    list_long_horizon_strategies,
    list_memory_governance_strategies,
    list_orchestration_profiles,
)

print('public api sample:', agentorch.__all__[:40], '...')
print('profiles:', list_orchestration_profiles())
print('context strategies:', list_context_strategies())
print('cooperation strategies:', list_cooperation_strategies())
print('long horizon strategies:', list_long_horizon_strategies())
print('memory governance strategies:', list_memory_governance_strategies())


public api sample: ['Agent', 'AgentCapability', 'AgentRegistry', 'AgentSpec', 'ArtifactRef', 'BaseRetriever', 'BeamSearchEvolutionAlgorithm', 'BraveSearchInput', 'ClassicRagConfig', 'ClassicRetriever', 'ChatPromptTemplate', 'DeliberativeRetriever', 'DeliberativeRagConfig', 'DeepResearchAgent', 'DeepResearchAgentConfig', 'Document', 'Coordinator', 'Context', 'CollectiveMemoryRecord', 'CollectiveMemoryMechanism', 'EpisodicCapsule', 'EpisodicSaliencePromotionPolicy', 'DocumentSection', 'ExecutionRequest', 'ExecutionResult', 'EvaluationResult', 'FeedbackDecision', 'FeedbackHandle', 'FeedbackKind', 'FeedbackSeverity', 'FeedbackStatus', 'FewShotExample', 'FewShotPromptCard', 'CotReasoning', 'EvolutionConfig', 'EvolutionAlgorithm', 'EvolutionExecutionContext', 'EvolutionManager', 'EvolutionRegistration', 'EvolutionRegistry'] ...
profiles: ['coding_agent', 'compact_single_agent', 'deep_research', 'default_safe', 'distributed_swarm', 'matriarchal_elephant', 'research_heavy', 'workflow_oriented'

## 4. 最小 Agent 运行

现在推荐把最常用的装配方式统一成：

- `Agent.acreate(...)` 直接在 notebook 里创建可运行 agent
- `model_config='gpt-4.1-mini'` 这种短写法直接可用
- `RuntimeConfig.agent(...)` 统一挂载 `reasoning / rag / prompt / strategy`
- `orchestration_profile` 负责一键展开推荐默认值
- `context_strategy` / `cooperation_strategy` 可以局部覆盖 profile

下面是 notebook 环境下的**最新最小推荐写法**。


In [7]:
from agentorch import Agent
from agentorch.config import RuntimeConfig

agent = await Agent.acreate(
    model_config='gpt-4.1-mini',
    config=RuntimeConfig.agent(
        system_prompt='你是一个清晰、准确、简洁的 agentorch 助手。',
        orchestration_profile='default_safe',
        context_strategy='compact',
        reasoning='react',
    ),
)

result = await agent.run(
    '请用三句话介绍 agentorch 这个包的作用。',
    thread_id='nb-basic-001',
)

print(result.output_text)
print('reasoning kind:', result.reasoning_kind)
print('resolved strategies:', result.reasoning_metadata.get('resolved_strategies'))
print('context budget:', result.reasoning_metadata.get('context_budget_report'))


Agentorch 是一个用于构建强化学习智能体的工具包，提供了多种算法的实现和训练框架。它支持分布式训练，方便大规模强化学习任务的开发。该包还具备灵活的模块设计，便于用户自定义和扩展智能体功能。
reasoning kind: react
resolved strategies: {'context': {'kind': 'compact', 'mode': 'compact', 'max_conversation_messages': 6, 'include_memory_summary': False, 'include_retrieval_summary': True, 'include_retrieval_evidence': False, 'include_retrieval_citations': True, 'include_retrieval_report': False, 'include_retrieval_plan': False, 'include_tool_descriptions': False, 'include_skill_instructions': True, 'include_task_packet': True, 'include_delegation_context': True, 'tool_result_policy': 'summary', 'tool_result_max_chars': 500, 'retrieval_evidence_max_items': 3, 'citation_max_items': 4, 'prompt_char_budget': 12000, 'prefer_system_compaction': True, 'budget_aware_compaction': False, 'salience_mode': 'off', 'salience_rerank_top_k': 8, 'segment_char_budget': None, 'segment_min_keep': 6, 'stage_attention_profiles': {}}, 'long_horizon': {'kind': 'long_running_safe', 'history_retention_policy

## 5. 结构化工具实验

推荐的新写法：

- 用 `ToolRegistry.from_tools(...)` 一次性创建工具注册表
- 再通过 `Agent.acreate(...)` 直接把 `tools` 装入 runtime
- notebook 里全程保持 async 风格



In [8]:
from pydantic import BaseModel
from agentorch import Agent, ToolRegistry, tool
from agentorch.config import RuntimeConfig


class AddInput(BaseModel):
    a: int
    b: int


@tool(description='Add two integers together.')
async def add_numbers(input: AddInput):
    return {'sum': input.a + input.b}


tools = ToolRegistry.from_tools(add_numbers)

agent = await Agent.acreate(
    model_config='gpt-4.1-mini',
    tools=tools,
    config=RuntimeConfig.agent(reasoning='react'),
)

result = await agent.run(
    '请调用 add_numbers 工具，计算 123 + 456，并解释结果。',
    thread_id='nb-tool-003',
)

print(result.output_text)
print(result.tool_results)



计算 123 + 456 的结果是 579。这是因为将两个整数相加时，将它们的数值合并得到一个新的整数。在本例中，123 和 456 相加得到的和是 579。
[ToolExecutionResult(tool_call_id='call_zb6T9VyOy0i5eGckzKSpt0zH', tool_name='add_numbers', output={'sum': 579}, is_error=False, error_message=None, duration=0.00017290000687353313, metadata={})]


### 如果你想复用旧 thread_id

先清理旧线程消息，避免旧状态残留：


In [ ]:
# 示例：
# await runtime.memory.clear_thread('nb-tool-001')


## 6. 查看工具调用的结构化结果


In [9]:
for item in result.tool_results:
    print(item.model_dump())


{'tool_call_id': 'call_zb6T9VyOy0i5eGckzKSpt0zH', 'tool_name': 'add_numbers', 'output': {'sum': 579}, 'is_error': False, 'error_message': None, 'duration': 0.00017290000687353313, 'metadata': {}}


### 6.1 一次回复触发多个工具调用

如果模型在同一次回复里规划出多个 `tool_calls`，`agentorch` 会把这些调用的执行结果统一收集到 `result.tool_results` 里。

下面这个例子要求模型在**同一次工具规划**里同时调用 `add_numbers` 和 `multiply_numbers`，便于观察多工具调用的结果聚合。


In [10]:
from pydantic import BaseModel
from agentorch import Agent, ToolRegistry, tool
from agentorch.config import RuntimeConfig


class MultiplyInput(BaseModel):
    a: int
    b: int


@tool(description='Multiply two integers together.')
async def multiply_numbers(input: MultiplyInput):
    return {'product': input.a * input.b}


multi_tools = ToolRegistry.from_tools(add_numbers, multiply_numbers)

multi_agent = await Agent.acreate(
    model_config='gpt-4.1-mini',
    tools=multi_tools,
    config=RuntimeConfig.agent(reasoning='react'),
)

multi_result = await multi_agent.run(
    '请严格按要求执行：在同一次回复里同时调用 add_numbers 和 multiply_numbers 两个工具，不要分两轮。先计算 12 + 34，再计算 12 * 34，最后用一句话总结两个结果。',
    thread_id='nb-tool-multi-001',
)

print(multi_result.output_text)
print('tool call count:', len(multi_result.tool_results))
print('tool names:', [item.tool_name for item in multi_result.tool_results])

for item in multi_result.tool_results:
    print(item.model_dump())


12 + 34 的结果是 46，12 * 34 的结果是 408，两个计算结果分别是 46 和 408。
tool call count: 2
tool names: ['add_numbers', 'multiply_numbers']
{'tool_call_id': 'call_KKvX8bDh1qM2B6ynyUPwhI11', 'tool_name': 'add_numbers', 'output': {'sum': 46}, 'is_error': False, 'error_message': None, 'duration': 0.00013639999815495685, 'metadata': {}}
{'tool_call_id': 'call_1JA2P5TrciZfDUi8D1Roibtz', 'tool_name': 'multiply_numbers', 'output': {'product': 408}, 'is_error': False, 'error_message': None, 'duration': 0.0001269999993382953, 'metadata': {}}


## 7. 多个工具调用稳定实验

如果你当前主要想验证“模型能不能在一次任务里调用多个工具”，最稳妥的方式是先不用 `python_interpreter`，而是改用几个纯 Python 函数工具。

这样可以避开本地沙箱、命令白名单、解释器路径等环境问题，把测试重点放在 `tool_calls` 和 `tool_results` 本身。


In [14]:
from pydantic import BaseModel
from agentorch import Agent, ToolRegistry, tool
from agentorch.config import RuntimeConfig


class AddInput(BaseModel):
    a: int
    b: int


@tool(description='Add two integers together.')
async def add_numbers(input: AddInput):
    return {'sum': input.a + input.b}


class MultiplyInput(BaseModel):
    a: int
    b: int


@tool(description='Multiply two integers together.')
async def multiply_numbers(input: MultiplyInput):
    return {'product': input.a * input.b}


class WeatherInput(BaseModel):
    city: str


@tool(description='Return a fake weather summary for demo/testing.')
async def get_weather(input: WeatherInput):
    mocked = {
        '上海': {'weather': '多云', 'temperature_c': 24},
        '北京': {'weather': '晴', 'temperature_c': 26},
        '深圳': {'weather': '小雨', 'temperature_c': 28},
    }
    return {'city': input.city, **mocked.get(input.city, {'weather': '未知', 'temperature_c': None})}


class CurrencyInput(BaseModel):
    amount_cny: float
    rate: float = 7.2


@tool(description='Convert CNY to USD with a mocked fixed exchange rate for demo/testing.')
async def convert_cny_to_usd(input: CurrencyInput):
    usd = round(input.amount_cny / input.rate, 2)
    return {'amount_cny': input.amount_cny, 'rate': input.rate, 'amount_usd': usd}


tools = ToolRegistry.from_tools(add_numbers, multiply_numbers, get_weather, convert_cny_to_usd)

agent = await Agent.acreate(
    model_config='gpt-4.1-mini',
    tools=tools,
    config=RuntimeConfig.agent(reasoning='react'),
)

result = await agent.run(
    '请在同一次回复中同时调用 3 个工具，不要分多轮：1) 调用 add_numbers 计算 25 + 17；2) 调用 multiply_numbers 计算 8 * 9；3) 调用 get_weather 查询银川5月1日的天气。最后把三个工具结果整理成三行输出。',
    thread_id='nb-multi-tools-002',
)

print(result.output_text)
print('tool call count:', len(result.tool_results))
print('tool names:', [item.tool_name for item in result.tool_results])
for item in result.tool_results:
    print(item.model_dump())


25 + 17 = 42
8 * 9 = 72
银川5月1日的天气：未知，暂无具体温度信息。
tool call count: 3
tool names: ['add_numbers', 'multiply_numbers', 'get_weather']
{'tool_call_id': 'call_qnTXpbtQSAWsIiN37dvRYsoF', 'tool_name': 'add_numbers', 'output': {'sum': 42}, 'is_error': False, 'error_message': None, 'duration': 0.00013159999798517674, 'metadata': {}}
{'tool_call_id': 'call_kvFnYySDfM3gSNXpzaM7zqSg', 'tool_name': 'multiply_numbers', 'output': {'product': 72}, 'is_error': False, 'error_message': None, 'duration': 0.00014910000027157366, 'metadata': {}}
{'tool_call_id': 'call_ghBMoNNdtHLdheuYNsEDmF2b', 'tool_name': 'get_weather', 'output': {'city': '银川', 'weather': '未知', 'temperature_c': None}, 'is_error': False, 'error_message': None, 'duration': 0.00017789999401429668, 'metadata': {}}


## 8. 再做一次 4 工具联合调用压测


In [15]:
stress_result = await agent.run(
    '请在同一次回复中连续调用 4 个工具，不要拆成多轮：1) add_numbers 计算 101 + 99；2) multiply_numbers 计算 7 * 11；3) get_weather 查询北京天气；4) convert_cny_to_usd 把 144 元人民币按默认汇率换算成美元。最后输出一个简短汇总。',
    thread_id='nb-multi-tools-003',
)

print(stress_result.output_text)
print('tool call count:', len(stress_result.tool_results))
print('tool names:', [item.tool_name for item in stress_result.tool_results])

for item in stress_result.tool_results:
    print(item.model_dump())


计算结果如下：
1) 101 + 99 = 200
2) 7 * 11 = 77
3) 北京天气：晴，气温26°C
4) 144元人民币按默认汇率7.2换算成美元约为20美元。

简短总结：加法得200，乘法得77，北京晴朗且温暖，人民币144元约合20美元。
tool call count: 4
tool names: ['add_numbers', 'multiply_numbers', 'get_weather', 'convert_cny_to_usd']
{'tool_call_id': 'call_CP5d5Zi8AIKV4RdYQUvLl1IS', 'tool_name': 'add_numbers', 'output': {'sum': 200}, 'is_error': False, 'error_message': None, 'duration': 0.0002910999974119477, 'metadata': {}}
{'tool_call_id': 'call_zX8KKWnc6OlzlSR9Ui9Ezc3H', 'tool_name': 'multiply_numbers', 'output': {'product': 77}, 'is_error': False, 'error_message': None, 'duration': 0.0002327999973203987, 'metadata': {}}
{'tool_call_id': 'call_8C6Dcw0LPCcaprfzi78xVBDI', 'tool_name': 'get_weather', 'output': {'city': '北京', 'weather': '晴', 'temperature_c': 26}, 'is_error': False, 'error_message': None, 'duration': 0.000294400000711903, 'metadata': {}}
{'tool_call_id': 'call_tqvbytF6yVkRetorzndzvMCh', 'tool_name': 'convert_cny_to_usd', 'output': {'amount_cny': 144.0, 'rate': 7.2, 'amo

## 9. Memory 实验

当前 memory 主要支持：

- thread message
- thread summary
- long-term record
- checkpoint
- clear_thread


In [16]:
from agentorch.memory import MemoryManager, MemoryRecord
from agentorch.core import Message

memory = MemoryManager()

await memory.append_message('thread-demo', Message(role='user', content='我偏好异步优先架构'))
await memory.append_message('thread-demo', Message(role='assistant', content='收到，我会按异步优先来设计。'))

summary = await memory.summarize_thread('thread-demo')
print(summary)

await memory.remember(
    MemoryRecord(
        thread_id='thread-demo',
        kind='preference',
        content='用户偏好异步优先架构',
        tags=['preference', 'architecture'],
    )
)

records = await memory.search(thread_id='thread-demo', query='异步')
records


user: 我偏好异步优先架构
assistant: 收到，我会按异步优先来设计。


[{'id': 15,
  'thread_id': 'thread-demo',
  'kind': 'preference',
  'content': '用户偏好异步优先架构',
  'tags': ['preference', 'architecture'],
  'metadata': {}},
 {'id': 25,
  'thread_id': 'thread-demo',
  'kind': 'preference',
  'content': '用户偏好异步优先架构',
  'tags': ['preference', 'architecture'],
  'metadata': {}},
 {'id': 1067,
  'thread_id': 'thread-demo',
  'kind': 'preference',
  'content': '用户偏好异步优先架构',
  'tags': ['preference', 'architecture'],
  'metadata': {'memory_role': 'individual',
   'confidence': 0.5,
   'source_agents': [],
   'status': 'candidate',
   'last_validated_at': None,
   'reuse_count': 0,
   'scope': None}},
 {'id': 4289,
  'thread_id': 'thread-demo',
  'kind': 'thread_message',
  'content': '我偏好异步优先架构',
  'tags': ['conversation', 'user'],
  'metadata': {'role': 'user',
   'name': None,
   'tool_call_id': None,
   'tool_calls': []}},
 {'id': 4290,
  'thread_id': 'thread-demo',
  'kind': 'thread_message',
  'content': '收到，我会按异步优先来设计。',
  'tags': ['conversation', 'assista

## 10. 清理线程上下文


In [17]:
await memory.clear_thread('thread-demo')
print(await memory.get_thread_messages('thread-demo'))


[Message(role='user', content='我偏好异步优先架构', name=None, tool_call_id=None, tool_calls=[], metadata={}), Message(role='assistant', content='收到，我会按异步优先来设计。', name=None, tool_call_id=None, tool_calls=[], metadata={})]


## 11. Workflow 基础实验

当前 workflow 除了基础节点，还已经支持更偏 RAG 编排的节点：

- `model`
- `tool`
- `router`
- `memory`
- `agent`
- `retrieve`
- `rag_router`
- `rag_mount`
- `rag_evaluate`

并且 workflow 节点可以局部覆盖：

- `rag_strategy` / `rag_mode`
- `context_strategy`
- `long_horizon_strategy`
- `cooperation_strategy`

先从最基础的 memory + model 流程开始。


In [18]:
from agentorch import Agent, WorkflowBuilder
from agentorch.config import RuntimeConfig
from agentorch.workflow import Node

workflow = (
    WorkflowBuilder()
    .then(
        Node(
            id='remember_project',
            kind='memory',
            config={
                'action': 'remember',
                'kind': 'project_note',
                'content': '`agentorch` 是一个代码优先、异步优先的 Python 智能体编排框架，用来构建可编程的 agent 系统。它提供结构化工具、工作流、RAG、记忆、推理策略、沙箱执行和多智能体委派能力。',
                'tags': ['project', 'overview'],
            },
        )
    )
    .then(Node.model_node('summarize', prompt='请总结 agentorch 的定位和适合的使用场景。'))
    .build()
)

agent = await Agent.acreate(
    model_config='gpt-4.1-mini',
    workflow=workflow,
    config=RuntimeConfig.workflow(reasoning='react'),
)

result = await agent.run('请开始执行这个 workflow。', thread_id='nb-workflow-basic-001')
print(result.output_text)



{"status": "completed", "output_text": "AgentOrch 是一个用于构建和管理智能代理工作流的框架。它的定位主要是作为协调和调度多个智能代理执行复杂任务的工具，帮助用户更好地设计、编排和监控代理之间的交互与协作。\n\n适合的使用场景包括：\n1. 多步骤、多代理的自动化工作流：需要多个智能代理协同完成的复杂任务。\n2. 任务编排与调度：对任务的执行顺序、依赖关系和状态管理有较高要求。\n3. 智能系统集成：将不同类型的智能模型和工具整合到一个统一的工作流中。\n4. 实时监控和管理智能代理的执行状态。\n5. 需要灵活扩展和定制的智能代理解决方案。\n\n总结：AgentOrch 适合用于需要多智能代理协作完成复杂自动化任务的应用场景，尤其是在任务编排和流程管理方面有较高需求的系统。"}


## 12. Multi-format Deliberative RAG 接口层实验

这里开始统一展示最新版 RAG 接口：

- `IndexedKnowledgeBase.acreate(...)`
- `KnowledgeAsset.from_path(...)`
- `RetrievalIntent.from_question(...)`
- `RagStrategyConfig.for_classic(...) / for_deliberative(...) / for_hybrid(...)`
- `deliberative_retrieve / search_knowledge_assets / open_retrieved_evidence`

默认推荐优先使用多格式、主动式检索链路，而不是只停留在最小 `chunk top-k` 示例。


In [ ]:
from agentorch import IndexedKnowledgeBase
from agentorch.knowledge import Document, RetrievalIntent

knowledge_base = await IndexedKnowledgeBase.acreate(
    documents=[
        Document(id='doc-1', text='agentorch is a code-first, async-first agent orchestration framework for Python.'),
        Document(id='doc-2', text='Deliberative RAG retrieves evidence with source routing, structure-aware extraction, and coverage checks.'),
        Document(id='doc-3', text='Hybrid RAG can combine classic coarse recall with deliberative evidence refinement.'),
    ]
)

report = await knowledge_base.get_retriever().retrieve_report(
    RetrievalIntent.from_question(
        'agentorch framework and deliberative rag',
        must_cover=['agentorch', 'deliberative'],
        max_documents=4,
    )
)

print('--- summary ---')
print(report.summary)
print('--- coverage ---')
print(report.coverage.model_dump())
print('--- visited sources ---')
print(report.visited_sources)
print('--- citations ---')
for item in report.citations:
    print(item.model_dump())


## 13. 把检索接入 Runtime

这里推荐直接把 RAG 和上下文治理一起装进 runtime：

- 用 `rag=` 选择 classic / deliberative / hybrid
- 用 `orchestration_profile=` 一次性装入推荐策略
- 用 `context_strategy=` 控制 evidence、citation、tool 描述是否进 prompt
- 运行后直接查看 `result.reasoning_metadata['resolved_strategies']`


In [ ]:
from agentorch import Agent, IndexedKnowledgeBase
from agentorch.config import RuntimeConfig
from agentorch.knowledge import Document, RagStrategyConfig

knowledge_base = await IndexedKnowledgeBase.acreate(
    documents=[
        Document(
            id='doc-1',
            text='agentorch is designed for code-first, async-first agent orchestration in Python.',
            metadata={'scopes': ['product']},
        ),
        Document(
            id='doc-2',
            text='The runtime can mount retrieved evidence into the current agent context and report uncovered items.',
            metadata={'scopes': ['product']},
        ),
    ]
)

agent = await Agent.acreate(
    model_config='gpt-4.1-mini',
    knowledge_base=knowledge_base,
    config=RuntimeConfig.agent(
        orchestration_profile='research_heavy',
        context_strategy='research_heavy',
        cooperation_strategy='matriarchal_elephant',
        rag=RagStrategyConfig.for_hybrid(
            knowledge_scope=['product'],
            must_cover=['agentorch', 'retrieved evidence'],
            max_steps=3,
        ),
        reasoning='react',
    ),
)

result = await agent.run(
    'What is agentorch designed for, and how does runtime retrieval work?',
    thread_id='nb-rag-001',
)

print(result.output_text)
print('resolved strategies:', result.reasoning_metadata.get('resolved_strategies'))
print('context budget:', result.reasoning_metadata.get('context_budget_report'))


## 14. AgentRegistry 实验

多智能体系统里，agent 需要先注册，再被 workflow 或 supervisor 调度。


In [2]:
from agentorch import AgentRegistry, AgentSpec

registry = AgentRegistry()
print(registry.list_specs())


[]


## 15. 构建一个 specialist agent 并注册

下面构建一个最小 specialist agent，并注册到 `AgentRegistry`。


In [3]:
from pydantic import BaseModel
from agentorch import Agent, AgentCapability, AgentRegistry, AgentSpec, ToolRegistry, tool
from agentorch.config import RuntimeConfig


class EchoInput(BaseModel):
    text: str


@tool(description='Echo a message as structured data.')
async def echo(input: EchoInput):
    return {'echo': input.text}


async def build_specialist_agent(description: str) -> Agent:
    return await Agent.acreate(
        model_config='gpt-4.1-mini',
        tools=ToolRegistry.from_tools(echo),
        config=RuntimeConfig.agent(
            system_prompt=description,
            reasoning='react',
        ),
    )


registry = AgentRegistry()
planner_agent = await build_specialist_agent('You are a planning specialist for decomposition tasks.')
registry.register(
    AgentSpec.assistant(
        'planner',
        description='Planning specialist for decomposition tasks',
        capabilities=[AgentCapability.PLAN, AgentCapability.TOOL_USE],
        tools=['echo'],
        knowledge_scopes=['architecture'],
        preferred_reasoning_kind='plan_execute',
    ),
    planner_agent,
)

print(registry.list_specs())



[AgentSpec(name='planner', description='Planning specialist for decomposition tasks', tags=[], input_schema={}, output_schema={}, capabilities=[<AgentCapability.PLAN: 'plan'>, <AgentCapability.TOOL_USE: 'tool_use'>], allowed_tools=['echo'], allowed_knowledge_scopes=['architecture'], max_delegation_depth=1, supports_parallel_tasks=False, policy_profile=AgentPolicyProfile(retrieval_mode='inline', reasoning_mode='react', allow_tool_calls=True, allow_delegation=False, default_rag_strategy=None, allowed_rag_modes=['classic', 'deliberative', 'hybrid'], allow_inline_rag_mount=True, allow_explicit_retrieval_tool=True, preferred_reasoning_kind='plan_execute', default_context_strategy=None, default_long_horizon_strategy=None, default_cooperation_strategy=None, notes={}), metadata={})]


## 16. Supervisor 多智能体动态委派实验


In [4]:
from agentorch import Agent, Supervisor

supervisor = Supervisor(registry=registry)
agent = await Agent.acreate(
    model_config='gpt-4.1-mini',
    agent_registry=registry,
    supervisor=supervisor,
)

result = await agent.run(
    'Please plan the implementation steps for a Python agent framework.',
    thread_id='nb-supervisor-001',
)

print(result.output_text)



[planner] To plan the implementation steps for a Python agent framework, I will outline the key phases and tasks involved. This plan will cover initial design, core component development, and testing.

Step 1: Define Requirements and Scope
- Determine the types of agents (e.g., reactive, deliberative, hybrid)
- Specify features (e.g., communication, learning, decision-making)
- Identify use cases and environment setup requirements

Step 2: Design the Architecture
- Define core components (Agent, Environment, Sensors, Actuators)
- Design interfaces and interaction protocols among components
- Decide on modularity and extensibility aspects

Step 3: Set Up Project Structure
- Create repository and directory layout
- Choose dependencies and tools (e.g., async libraries, ML frameworks)
- Setup virtual environment and initial files

Step 4: Implement Core Components
- Develop base Agent class with common properties and methods
- Implement Environment simulation or interface
- Create Sensor a

## 17. Workflow 中的 agent 节点实验

这条链路演示静态 DAG 里的 agent 节点委派。


In [5]:
from agentorch import Agent, Workflow
from agentorch.config import RuntimeConfig
from agentorch.workflow import Node

workflow = Workflow.chain(
    Node.agent(
        'delegate',
        'planner',
        goal='Please plan the steps for building an async Python agent runtime.',
        output_key='planner_output',
    )
)

agent = await Agent.acreate(
    model_config='gpt-4.1-mini',
    agent_registry=registry,
    workflow=workflow,
    config=RuntimeConfig.workflow(reasoning='react'),
)

result = await agent.run('ignored by workflow node config', thread_id='nb-workflow-agent-001')
print(result.output_text)



{"status": "completed", "output_text": "To plan the steps for building an async Python agent runtime, I will break down the process into key components and development stages:\n\n1. Define the Requirements and Architecture:\n   - Specify the core functionalities of the async agent runtime (e.g., task scheduling, concurrency management, communication interfaces).\n   - Choose the asynchronous framework/library (e.g., asyncio, Trio, or curio).\n   - Define the agent model: types of agents, their states, and lifecycle.\n   - Outline the runtime architecture including event loop, task queue, and inter-agent communication.\n\n2. Set Up the Project Environment:\n   - Create the Python project structure.\n   - Set up dependencies including chosen asyncio framework.\n   - Configure testing, linting, and development tools.\n\n3. Implement Core Async Runtime Components:\n   - Build event loop integration with the async framework.\n   - Develop task scheduler to manage agent actions and prioritiz

## 18. 当前完整装配模板

这一节展示当前版本比较完整的装配方式：

- `orchestration_profile`
- `reasoning + rag + context_strategy`
- `tools + sandbox`
- `knowledge_base`
- `agent_registry + supervisor`

也就是你现在做研究型智能体或多智能体编排时最接近真实项目的一种装配方式。


In [1]:
import json
from pathlib import Path
from pprint import pprint

from agentorch import Agent, IndexedKnowledgeBase, SandboxManager, ToolRegistry
from agentorch.config import RuntimeConfig
from agentorch.knowledge import Document, RagStrategyConfig
from agentorch.sandbox import SandboxPolicy


# -----------------------------
# 1. 创建沙箱
# -----------------------------
sandbox = SandboxManager(
    policy=SandboxPolicy(
        allowed_paths=[Path.cwd()],
        command_allowlist=["python", "git", "powershell", "cmd"],
        timeout=15.0,
    )
)

# -----------------------------
# 2. 创建工具注册表
# -----------------------------
tools = ToolRegistry.with_bundles(
    workspace_root=Path.cwd(),
    sandbox=sandbox,
)

# -----------------------------
# 3. 创建知识库
# -----------------------------
knowledge_base = await IndexedKnowledgeBase.acreate(
    documents=[
        Document(
            id="doc-1",
            text="agentorch supports runtime, tools, memory, workflows, reasoning, and multi-agent orchestration.",
            metadata={"scopes": ["overview"]},
        ),
        Document(
            id="doc-2",
            text="Deliberative RAG can route sources and validate coverage before returning evidence.",
            metadata={"scopes": ["overview", "rag"]},
        ),
    ]
)

# -----------------------------
# 4. 创建运行配置
# -----------------------------
# 为了更容易观察工具调用过程，这里用相对轻一点的配置
runtime_config = RuntimeConfig.agent(
    reasoning="react",
    context_strategy="compact",
    rag=RagStrategyConfig.for_deliberative(
        knowledge_scope=["overview"],
        max_steps=1,
    ),
    max_steps=4,
)

# -----------------------------
# 5. 创建 Agent
# -----------------------------
agent = await Agent.acreate(
    model_config="gpt-4.1-mini",
    tools=tools,
    sandbox=sandbox,
    knowledge_base=knowledge_base,
    config=runtime_config,
)

# -----------------------------
# 6. 给底层模型 generate 打补丁
#    打印每轮模型 token 消耗
# -----------------------------
call_logs = []
original_generate = agent.runtime.model.generate

async def traced_generate(request):
    call_index = len(call_logs) + 1

    response = await original_generate(request)

    usage = response.usage
    item = {
        "call_index": call_index,
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "finish_reason": response.finish_reason,
        "tool_calls": len(response.tool_calls),
        "content_preview": (response.content or "")[:200],
    }
    call_logs.append(item)

    print(f"\n========== 模型调用 #{call_index} ==========")
    print(f"输入 token: {usage.prompt_tokens}")
    print(f"输出 token: {usage.completion_tokens}")
    print(f"总 token: {usage.total_tokens}")
    print(f"finish_reason: {response.finish_reason}")
    print(f"tool_calls 数量: {len(response.tool_calls)}")

    if response.tool_calls:
        print("工具调用计划:")
        for tc in response.tool_calls:
            print(f"  tool_name: {tc.name}")
            print(f"  tool_call_id: {tc.id}")
            print(f"  arguments:")
            pprint(tc.arguments)

    if response.content:
        print("模型输出预览:")
        print(response.content[:300])

    return response

agent.runtime.model.generate = traced_generate


# -----------------------------
# 7. 流式运行，打印全过程事件
# -----------------------------
final_result = None

async for event in agent.run(
    "请先用一句话介绍你自己，然后查看当前目录有哪些文件，并简要列出来。",
    thread_id="tool-trace-1",
    stream=True,
):
    print(f"\n==================== 事件: {event.event_type} ====================")

    # 你最关心的是 payload
    if event.payload:
        pprint(event.payload)

    # 如果事件里带有工具调用信息
    if event.tool_calls:
        print("tool_calls:")
        for tc in event.tool_calls:
            print(f"  tool_name: {tc.name}")
            print(f"  tool_call_id: {tc.id}")
            print(f"  arguments:")
            pprint(tc.arguments)

    # 流式文本增量
    if event.delta_text:
        print("delta_text:")
        print(event.delta_text)

    # 最终结果事件
    if event.event_type == "final_result" and event.result is not None:
        final_result = event.result


# -----------------------------
# 8. 打印最终结果
# -----------------------------
print("\n\n========== 最终输出 ==========")
if final_result is not None:
    print(final_result.output_text)

    print("\n========== 最终总 token ==========")
    print(f"总输入 token: {final_result.usage.prompt_tokens}")
    print(f"总输出 token: {final_result.usage.completion_tokens}")
    print(f"总 token: {final_result.usage.total_tokens}")

    print("\n========== 最终工具结果 ==========")
    for i, tr in enumerate(final_result.tool_results, start=1):
        print(f"\n工具 #{i}")
        print(f"tool_name: {tr.tool_name}")
        print(f"is_error: {tr.is_error}")
        print(f"error_message: {tr.error_message}")
        print(f"duration: {tr.duration}")
        print("output:")
        pprint(tr.output)


# -----------------------------
# 9. 打印逐轮模型调用统计
# -----------------------------
print("\n========== 每轮模型调用 token 明细 ==========")
for item in call_logs:
    print(
        f"第 {item['call_index']} 轮 | "
        f"输入: {item['prompt_tokens']} | "
        f"输出: {item['completion_tokens']} | "
        f"总计: {item['total_tokens']} | "
        f"finish_reason: {item['finish_reason']} | "
        f"tool_calls: {item['tool_calls']}"
    )

# -----------------------------
# 10. 关闭资源
# -----------------------------
await agent.aclose()



==================== 事件: run_started ====================
{'metadata': {}, 'trace_id': '1ac73f58-a906-429b-a9dd-b03f993fea03'}

==================== 事件: task_created ====================
{'metadata': {}, 'trace_id': '1ac73f58-a906-429b-a9dd-b03f993fea03'}

==================== 事件: retrieval_started ====================
{'metadata': {'resolved_skill_routing': {'disclosure_level': 'progressive',
                                         'include_allowed_tools': True,
                                         'max_active': 2,
                                         'max_candidates': 4,
                                         'mode': 'progressive'},
              'selected_skill_routes': []},
 'query': '请先用一句话介绍你自己，然后查看当前目录有哪些文件，并简要列出来。',
 'trace_id': '1ac73f58-a906-429b-a9dd-b03f993fea03'}

==================== 事件: retrieval_completed ====================
{'chunk_count': 4,
 'knowledge_scope': ['overview'],
 'metadata': {'memory_recall_report': {'config': {'allow_cross_thread_recall': Fa

In [ ]:
import json

log_path = "tool_trace_log.jsonl"

final_result = None

with open(log_path, "w", encoding="utf-8") as f:
    async for event in agent.run(
        "请先用一句话介绍你自己，然后查看当前目录有哪些文件，并简要列出来。",
        thread_id="tool-trace-file",
        stream=True,
    ):
        record = {
            "event_type": event.event_type,
            "payload": event.payload,
            "delta_text": event.delta_text,
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

        if event.event_type == "final_result" and event.result is not None:
            final_result = event.result

print("日志已保存到:", log_path)


APIConnectionError: Connection error.

: 

## 19. 推理框架总览

当前版本已经支持通过统一工厂 API 创建五种推理框架：

- `CoT`：线性分步推理
- `ReAct`：推理与工具行动交替
- `Plan-and-Execute`：先规划再执行
- `ToT`：多分支候选探索与剪枝
- `Reflexion`：尝试、反思、再尝试

这些框架都可以通过 `create_reasoning_framework(...)` 创建，并直接传给 `Runtime(policy=...)`。


In [6]:
from agentorch import ReasoningStrategyConfig, create_reasoning_framework

strategy_configs = [
    ReasoningStrategyConfig.cot(config={'max_steps': 6}),
    ReasoningStrategyConfig.react(config={'max_steps': 8}),
    ReasoningStrategyConfig.plan_execute(config={'max_planning_steps': 5, 'max_execution_steps': 8}),
    ReasoningStrategyConfig.tot(config={'branch_factor': 3, 'max_depth': 3, 'top_k': 2}),
    ReasoningStrategyConfig.reflexion(config={'max_attempts': 3, 'enable_self_reflection': True}),
]

for strategy in strategy_configs:
    framework = create_reasoning_framework(strategy.kind, **strategy.config)
    print(strategy.model_dump())
    print(framework.config)
    print('-' * 80)



{'kind': <ReasoningKind.COT: 'cot'>, 'config': {'max_steps': 6}, 'allow_tool_calls': False, 'allow_retrieval': True, 'allow_delegation': False}
kind=<ReasoningKind.COT: 'cot'> max_steps=6 expose_trace=True allow_tool_calls=False
--------------------------------------------------------------------------------
{'kind': <ReasoningKind.REACT: 'react'>, 'config': {'max_steps': 8}, 'allow_tool_calls': True, 'allow_retrieval': True, 'allow_delegation': False}
kind=<ReasoningKind.REACT: 'react'> max_steps=8 expose_trace=True
--------------------------------------------------------------------------------
{'kind': <ReasoningKind.PLAN_EXECUTE: 'plan_execute'>, 'config': {'max_planning_steps': 5, 'max_execution_steps': 8}, 'allow_tool_calls': True, 'allow_retrieval': True, 'allow_delegation': False}
kind=<ReasoningKind.PLAN_EXECUTE: 'plan_execute'> max_steps=8 expose_trace=True max_planning_steps=5 max_execution_steps=8 allow_replan=True
-----------------------------------------------------------

## 20. CoT 实验

适合做结构化解释、数学推导、设计拆解等不强依赖工具交互的任务。

运行后可以同时查看：

- `result.output_text`
- `result.reasoning_kind`
- `result.reasoning_trace`


In [4]:
from agentorch import Agent
from agentorch.config import RuntimeConfig

agent = await Agent.acreate(
    model_config='gpt-4.1-mini',
    config=RuntimeConfig.agent(
        reasoning='cot',
        system_prompt='你是一个底层架构分析师。',
    ),
)

result = await agent.run(
    '请分步骤分析，一个底层智能体编排框架为什么要有 model、runtime、tools、memory、workflow 这几个层。',
    thread_id='nb-reasoning-cot-001',
)

print('reasoning kind:', result.reasoning_kind)
print('--- output ---')
print(result.output_text)
print('--- reasoning trace ---')
print(result.reasoning_trace)



reasoning kind: cot
--- output ---
好的，下面我将分步骤分析为什么一个底层智能体编排框架需要包含 model、runtime、tools、memory、workflow 这几个层。

## 分步骤分析

### 1. Model 层——智能体的认知与推理基础
- **角色**：提供智能体的核心认知能力，如语言理解、推理、决策等。
- **原因**：
  - 智能体需要依赖预训练模型或定制模型来完成自然语言理解、知识抽取、任务规划等认知任务。
  - 这是智能体行为的核心“思维引擎”，没有这一层，智能体无法“理解”输入，也无法生成合理输出。
- **功能**：封装模型调用接口，支持模型版本管理与模型替换。

### 2. Runtime 层——模型执行与资源调度的运行时环境
- **角色**：负责在实际运行时调度和执行模型及其他计算资源。
- **原因**：
  - 模型运行需要硬件资源管理（如GPU调度）、并发控制、异步处理等。
  - 需要保证模型推理的效率和稳定性。
  - 动态调整计算资源以适应不同任务负载。
- **功能**：管理模型的加载、执行、卸载；处理调用的效率和容错。

### 3. Tools 层——智能体调用的辅助工具集
- **角色**：提供模型以外的辅助能力，如搜索工具、计算器、API调用等。
- **原因**：
  - 智能体单纯依赖模型可能难以完成复杂任务（如实时数据获取、外部系统交互）。
  - 需要集成丰富的外部能力，增强智能体的实际应用范围。
- **功能**：封装各种工具接口，实现模型与外部能力的桥接。

### 4. Memory 层——智能体的状态保持与上下文存储
- **角色**：用于存储智能体的上下文信息、历史对话、任务状态等。
- **原因**：
  - 智能体需要保持上下文连续性，理解多轮对话及任务进展。
  - 通过记忆库，智能体能够回溯历史，提升交互质量。
- **功能**：持久存储、多轮上下文管理，支持状态恢复。

### 5. Workflow 层——智能体任务流程编排引擎
- **角色**：设计和执行复杂任务的步骤与逻辑，实现多模型、多工具、多记忆的协同。
- **原因**：
  - 智能体实际应用往往涉及多个子任务和协同调用，单一步骤模型调用不足。
  - 需要编排模型推理

## 21. Plan-and-Execute / ToT / Reflexion 快速实验

这三个更适合研究型编排：

- `Plan-and-Execute`：任务链长、流程明确
- `ToT`：需要比较多个候选方案
- `Reflexion`：希望 agent 先尝试，再自我改进


In [ ]:
from agentorch import Agent
from agentorch.config import RuntimeConfig

frameworks = [
    ('plan_execute', '请先规划，再给出一个 Python 智能体系统的实现步骤。'),
    ('tot', '请比较三种多智能体协作架构，并择优给出推荐。'),
    ('reflexion', '请先给出一个方案，再自我检查并改进它。'),
]

for name, prompt in frameworks:
    agent = await Agent.acreate(
        model_config='gpt-4.1-mini',
        config=RuntimeConfig.agent(reasoning=name),
    )
    result = await agent.run(prompt, thread_id=f'nb-{name}-001')
    print('=' * 80)
    print('framework:', name)
    print('output:')
    print(result.output_text)
    print('trace:')
    print(result.reasoning_trace)



## 22. 架构设计案例一：单 Agent 架构分析师

这个案例适合你做“架构顾问型 agent”：

- 用 `Plan-and-Execute` 做设计拆解
- 用 `RAG` 注入你的设计规范或技术文档
- 用 `orchestration_profile='research_heavy'` 打开更重的上下文与证据注入
- 输出最终方案 + 推理过程 + 已解析策略


In [ ]:
from agentorch import Agent, IndexedKnowledgeBase
from agentorch.config import RuntimeConfig
from agentorch.knowledge import Document, RagStrategyConfig

design_kb = await IndexedKnowledgeBase.acreate(
    documents=[
        Document(id='arch-1', text='底层智能体框架应分离模型层、工具层、推理层、运行时层与观测层。', metadata={'scopes': ['architecture']}),
        Document(id='arch-2', text='多智能体系统建议优先使用结构化任务包与共享工件，而不是无限自由对话。', metadata={'scopes': ['architecture', 'multi-agent']}),
    ]
)

agent = await Agent.acreate(
    model_config='gpt-4.1-mini',
    knowledge_base=design_kb,
    config=RuntimeConfig.agent(
        reasoning='plan_execute',
        rag=RagStrategyConfig.for_deliberative(
            knowledge_scope=['architecture'],
            must_cover=['模型层', '运行时层'],
            max_steps=3,
        ),
    ),
)

result = await agent.run(
    '请基于已有知识，设计一个单 Agent 架构分析师的系统骨架，并说明为什么这样分层。',
    thread_id='nb-arch-single-001',
)

print(result.output_text)



## 23. 架构设计案例二：RAG + 多智能体架构设计

这一节把“不同 specialist 绑定不同推理框架”落成一个最小案例：

- `planner`：`Plan-and-Execute`
- `researcher`：`ReAct`
- `critic`：`Reflexion`
- `supervisor`：统一调度 specialist
- `cooperation_strategy`：可以在 `matriarchal_elephant / distributed_herd / hybrid_herd` 间切换

为了避免 notebook 里因为关键词路由不稳定而只命中一个 agent，这里显式定义一个固定路由策略：

- 先调 `planner`
- 再调 `researcher`
- 最后调 `critic`


In [ ]:
from agentorch import Agent, AgentCapability, AgentRegistry, AgentSpec, IndexedKnowledgeBase, Supervisor
from agentorch.config import RuntimeConfig
from agentorch.knowledge import Document, RagStrategyConfig

shared_kb = await IndexedKnowledgeBase.acreate(
    documents=[
        Document(id='arch-1', text='底层智能体框架应分离模型层、工具层、推理层、运行时层与观测层。', metadata={'scopes': ['architecture']}),
        Document(id='arch-2', text='多智能体系统建议优先使用结构化任务包、共享工件与 supervisor 路由。', metadata={'scopes': ['architecture', 'multi-agent']}),
        Document(id='arch-3', text='Hybrid RAG can combine classic coarse recall with deliberative evidence extraction.', metadata={'scopes': ['architecture', 'rag']}),
    ]
)

planner = await Agent.acreate(
    model_config='gpt-4.1-mini',
    knowledge_base=shared_kb,
    config=RuntimeConfig.agent(
        reasoning='plan_execute',
        rag=RagStrategyConfig.for_hybrid(knowledge_scope=['architecture']),
    ),
)

reviewer = await Agent.acreate(
    model_config='gpt-4.1-mini',
    knowledge_base=shared_kb,
    config=RuntimeConfig.agent(
        reasoning='reflexion',
        rag=RagStrategyConfig.for_deliberative(knowledge_scope=['architecture', 'rag']),
    ),
)

registry = AgentRegistry()
registry.register(
    AgentSpec.assistant(
        'planner',
        description='Architecture planning specialist',
        capabilities=[AgentCapability.PLAN],
        knowledge_scopes=['architecture'],
        default_rag_strategy='hybrid',
        preferred_reasoning_kind='plan_execute',
    ),
    planner,
)
registry.register(
    AgentSpec.assistant(
        'reviewer',
        description='Architecture review specialist',
        capabilities=[AgentCapability.REVIEW],
        knowledge_scopes=['architecture', 'rag'],
        default_rag_strategy='deliberative',
        preferred_reasoning_kind='reflexion',
    ),
    reviewer,
)

orchestrator = await Agent.acreate(
    model_config='gpt-4.1-mini',
    agent_registry=registry,
    supervisor=Supervisor(registry=registry),
    knowledge_base=shared_kb,
    config=RuntimeConfig.agent(
        reasoning='react',
        rag=RagStrategyConfig.for_deliberative(knowledge_scope=['architecture']),
    ),
)

result = await orchestrator.run(
    '请给出一个 RAG + 多智能体 的底层架构设计，并指出 planner 与 reviewer 如何分工。',
    thread_id='nb-arch-multi-001',
)

print(result.output_text)



## 24. Deep Research 端到端实验（elephant + nutcracker + supervisor_aggregate）

这一节把当前最新的 deep research 能力同步到 notebook：

- `orchestration_profile='deep_research'`
- `matriarchal_elephant` 协作拓扑
- `nutcracker_memory` 长期记忆治理
- relay 友好的模型重试 / 节流参数
- `supervisor_aggregate` 顶层 reasoning metadata
- `child_reasoning` 摘要打印，直接查看每个 specialist 的 `resolved_strategies / context_budget_report / memory_recall_report`

这一版 notebook 复用 `examples/deep_research_agent.py` 中已经验证过的当前实现。


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Jupyter 容易复用旧模块缓存；先清掉本项目相关模块，再走一次新导入。
for name in list(sys.modules):
    if name == "agentorch" or name.startswith("agentorch.") or name == "examples.deep_research_agent":
        sys.modules.pop(name, None)

import agentorch
print("agentorch file:", agentorch.__file__)
print("has CooperationStrategyConfig:", hasattr(agentorch, "CooperationStrategyConfig"))

from examples.deep_research_agent import main as deep_research_main

# 直接运行当前最新版 deep research demo。
# 你会看到：
# 1. 顶层 Example smoke response
# 2. supervisor_aggregate
# 3. child_reasoning 摘要（每个 specialist 的策略 / context budget / memory recall）
await deep_research_main(stream=True)


agentorch file: c:\Users\24260\Desktop\研究生生涯\智能体开发范式\agentorch\__init__.py
has CooperationStrategyConfig: True
Resolved preset configuration:
{'orchestration_profile': 'deep_research', 'context_strategy': 'compact', 'long_horizon_strategy': 'long_running_safe', 'cooperation_strategy': 'matriarchal_elephant', 'memory_governance_strategy': 'nutcracker_memory', 'memory_policy_bundle': {'promotion_policy': 'episodic_salience', 'index_policy': 'scene_hash', 'recall_policy': 'scene_first', 'decay_policy': 'relevance_only'}, 'rag_mode': 'hybrid', 'memory_record_path': 'c:\\Users\\24260\\Desktop\\研究生生涯\\智能体开发范式\\.agentorch\\deep_research_records.db'}
Architecture linkage:
- elephant_context: handoff + collective sharing + prompt compaction
- nutcracker_memory: episodic promotion + scene indexing + long-term recall
Available tools:
- python_interpreter
- deliberative_retrieve
- search_knowledge_assets
- open_retrieved_evidence
Web search configured: True
Web search health: {'available': False, 

: 

## 25. 设计建议

如果你要继续把这个框架往研究或工程底盘推进，当前更值得优先增强的是：

- 把 notebook / README / examples 全部统一到 `orchestration_profile + strategy` 写法
- 继续扩展多格式 RAG ingestion 与 evidence opening 能力
- 给 Deep Research preset 增加更丰富的 web / skill / artifact 协同模板
- 把用户自定义 strategy / profile 注册做成更稳定的企业级接口
- 增强 supervisor 的 agent 选择策略与协作拓扑可视化
- 增强 observability，把 `resolved_strategies / context_budget / cooperation_report` 做成更好读的报告


## 26. 实验排错清单

如果再次遇到问题，优先按这个顺序检查：

1. notebook 里是否误用了 `run_sync()`
2. 是否修改过本地代码但没有重启 kernel
3. 是否复用了旧的 `thread_id`
4. `.env` 是否被正确读取
5. `base_url` 是否被规整成 `/v1`
6. sandbox 的 `allowed_paths` 和 `command_allowlist` 是否允许当前执行
7. 多 agent 实验前是否先把 specialist agent 注册到 `AgentRegistry`
8. RAG 实验是否通过 `RuntimeConfig.agent(...)` 或 `RuntimeConfig.workflow(...)` 正确装入 `rag=`
9. 是否显式用了 `orchestration_profile`，但又被局部 `context_strategy / cooperation_strategy` 覆盖
10. 如果跑 pytest，是否设置了 `PYTEST_DISABLE_PLUGIN_AUTOLOAD=1` 来避免本机全局插件污染

最常见的几条仍然是：

- notebook 要用 `await agent.run(...)`
- tool / multi-agent 调试失败时，请换新的 `thread_id` 或先 `clear_thread()`
- 遇到上下文注入和预期不一致时，先看 `result.reasoning_metadata['resolved_strategies']`


## 27. 联网搜索工具 API 参数测试

这一节专门验证三件事：

- `Runtime.acreate(...)` 是否可以通过 API 参数直接启用联网搜索工具
- `DeepResearchAgent.acreate(...)` 是否可以通过 `config['web_search']` 启用联网搜索，同时保留用户自定义工具
- 单个智能体是否可以直接通过 `brave_search` 完成一次网页搜索调用

说明：

- API key 直接从项目根目录 `.env` 读取
- 不会打印完整 key，只打印是否读取成功
- 这一节是“活测试”，会真正调用 Brave Search
- 如果当前网络环境无法连通 Brave，会输出诊断信息而不是直接让整个 cell 失败



In [ ]:
import os
import sys
from pathlib import Path

from pydantic import BaseModel

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


def load_env_file(env_path: Path) -> dict[str, str]:
    values: dict[str, str] = {}
    if not env_path.exists():
        return values
    for raw_line in env_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_values = load_env_file(PROJECT_ROOT / '.env')
brave_api_key = env_values.get('BRAVE_API_KEY') or os.getenv('BRAVE_API_KEY') or os.getenv('BRAVE_SEARCH_API_KEY')
print('BRAVE_API_KEY loaded:', bool(brave_api_key))
assert brave_api_key, '未在 .env 或环境变量中找到 BRAVE_API_KEY'

# 清掉旧模块缓存，确保 notebook 读取的是当前最新本地代码。
for name in list(sys.modules):
    if name == 'agentorch' or name.startswith('agentorch.'):
        sys.modules.pop(name, None)

from agentorch import Agent, DeepResearchAgent, Runtime, ToolRegistry, create_brave_search_tool, tool
from agentorch.core import Message, ModelRequest, ModelResponse, ToolCall, UsageInfo
from agentorch.knowledge import Document
from agentorch.models.base import BaseModelAdapter
from agentorch.tools import ToolError


class SimpleEchoModel(BaseModelAdapter):
    async def generate(self, request: ModelRequest) -> ModelResponse:
        return ModelResponse(
            message=Message(role='assistant', content='tooling test completed'),
            content='tooling test completed',
            finish_reason='stop',
            usage=UsageInfo(total_tokens=1),
        )


class SingleAgentWebSearchModel(BaseModelAdapter):
    def __init__(self) -> None:
        self.calls = 0

    async def generate(self, request: ModelRequest) -> ModelResponse:
        self.calls += 1
        if self.calls == 1:
            tool_call = ToolCall(
                id='call-brave-1',
                name='brave_search',
                arguments={
                    'query': 'multi-agent systems architecture',
                    'count': 2,
                    'search_lang': 'en',
                    'country': 'US',
                },
            )
            return ModelResponse(
                message=Message(role='assistant', content='Using brave_search to look up the web.', tool_calls=[tool_call]),
                content='Using brave_search to look up the web.',
                tool_calls=[tool_call],
                finish_reason='tool_calls',
                usage=UsageInfo(total_tokens=1),
            )

        latest_tool_message = next((m.content for m in reversed(request.messages) if m.role == 'tool'), '')
        return ModelResponse(
            message=Message(role='assistant', content=f'Single-agent web search completed. Tool payload summary: {latest_tool_message[:400]}'),
            content=f'Single-agent web search completed. Tool payload summary: {latest_tool_message[:400]}',
            finish_reason='stop',
            usage=UsageInfo(total_tokens=1),
        )


class EchoInput(BaseModel):
    text: str


@tool(description='Return a tagged echo for notebook testing.')
async def notebook_echo(input: EchoInput):
    return {'echo': input.text, 'source': 'custom_tool'}


async def safe_tool_execute(registry: ToolRegistry, name: str, arguments: dict) -> dict:
    try:
        result = await registry.execute(name, arguments)
        return {'ok': True, 'data': result.data}
    except ToolError as exc:
        return {'ok': False, 'error': str(exc)}


def print_search_summary(title: str, payload: dict) -> None:
    print(f'\n{title}')
    if not payload.get('ok'):
        print('search failed:', payload.get('error'))
        return
    results = payload.get('data', {}).get('results', [])
    print('result count:', len(results))
    for item in results[:3]:
        print('-', item.get('title'))
        print('  ', item.get('url'))


print('\n[1] 直接测试 brave_search 工具')
brave_registry = ToolRegistry.empty()
brave_registry.register(create_brave_search_tool(api_key=brave_api_key, timeout=60.0))
brave_payload = await safe_tool_execute(
    brave_registry,
    'brave_search',
    {
        'query': 'multi-agent systems architecture',
        'count': 3,
        'search_lang': 'en',
        'country': 'US',
    },
)
print_search_summary('[1] brave_search direct call', brave_payload)


print('\n[2] Runtime.acreate API 参数测试')
runtime = await Runtime.acreate(
    model=SimpleEchoModel(),
    include_web_tools=True,
    web_search_api_key=brave_api_key,
    custom_tools=[notebook_echo],
    knowledge_documents=[
        Document(
            id='runtime-web-1',
            text='Runtime should accept web tools and custom tools through API parameters.',
            metadata={'scopes': ['research']},
        )
    ],
)
print('runtime tools contain brave_search:', 'brave_search' in runtime.tools)
print('runtime tools contain notebook_echo:', 'notebook_echo' in runtime.tools)
assert 'brave_search' in runtime.tools
assert 'notebook_echo' in runtime.tools

runtime_payload = await safe_tool_execute(
    runtime.tools,
    'brave_search',
    {
        'query': 'agent orchestration supervisor worker',
        'count': 2,
        'search_lang': 'en',
        'country': 'US',
    },
)
print_search_summary('[2] runtime brave_search', runtime_payload)


print('\n[3] DeepResearchAgent.acreate API 参数测试')
agent = await DeepResearchAgent.acreate(
    model=SimpleEchoModel(),
    knowledge_documents=[
        Document(
            id='deep-web-1',
            text='DeepResearchAgent should support API-configured web search and user-defined tools.',
            metadata={'scopes': ['research']},
        )
    ],
    custom_tools=[notebook_echo],
    config={
        'knowledge_scope': ['research'],
        'web_search': {
            'provider': 'brave',
            'enabled': True,
            'api_key': brave_api_key,
        },
    },
)
print('agent tools contain brave_search:', 'brave_search' in agent.runtime.tools)
print('agent tools contain notebook_echo:', 'notebook_echo' in agent.runtime.tools)
assert 'brave_search' in agent.runtime.tools
assert 'notebook_echo' in agent.runtime.tools

agent_payload = await safe_tool_execute(
    agent.runtime.tools,
    'brave_search',
    {
        'query': 'planner executor critic multi-agent',
        'count': 2,
        'search_lang': 'en',
        'country': 'US',
    },
)
print_search_summary('[3] deep research agent brave_search', agent_payload)

custom_result = await agent.runtime.tools.execute('notebook_echo', {'text': 'web + custom tools ok'})
print('custom tool result:', custom_result.data)
assert custom_result.data['source'] == 'custom_tool'


print('\n[4] 单个智能体直接调用网页搜索')
single_agent_runtime = await Runtime.acreate(
    model=SingleAgentWebSearchModel(),
    include_web_tools=True,
    web_search_api_key=brave_api_key,
)
single_agent = Agent(runtime=single_agent_runtime)
try:
    single_agent_result = await single_agent.run('Please search the web for multi-agent architecture examples.', thread_id='single-agent-web-search-demo')
    print(single_agent_result.output_text)
except Exception as exc:
    print('single-agent web search failed:', type(exc).__name__, str(exc))


print('\n实验结束。如果上面出现 connect timeout，一般不是 API 参数接入问题，而是当前环境到 Brave 的网络连通性问题。')

